<a href="https://colab.research.google.com/github/lmassaron/fine-tuning-workshop/blob/main/05_generate_sentiment_explanations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Generating Financial Sentiment Explanations
In this notebook, we move beyond simple sentiment labels to create a **reasoned dataset**. Instead of just knowing *if* a headline is positive or negative, we want to understand *why*. 

### Why generate explanations?
- **Chain-of-Thought (CoT) Fine-tuning**: Training a model to reason before providing a label significantly improves its accuracy and robustness.
- **Transparency**: In financial applications, an "explainable" AI is more trustworthy for human analysts.
- **Data Augmentation**: We use a powerful teacher model (**Qwen 2.5 7B**) to "teach" a smaller student model how to think like a senior financial analyst.

### Technical Approach
We utilize **4-bit NormalFloat (NF4) quantization** to run a 7-billion parameter model on a single consumer GPU (e.g., RTX 3090/4090 or Google Colab T4/L4), ensuring high performance with manageable hardware requirements.

In [ ]:
%%capture
# Install necessary libraries for model training and evaluation
import os
if "COLAB_" not in "".join(os.environ.keys()):
    pass
else:
    !pip install -U transformers trl accelerate bitsandbytes

### Environment Setup

We import standard data science and deep learning libraries. Note the use of `bitsandbytes` for quantization and `tqdm` for progress tracking during long-running inference tasks.

In [ ]:
import torch
from datasets import load_dataset, Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm.auto import tqdm
import os

## 1. Configuration

We define the core parameters for our generation pipeline. 

| Parameter | Value | Description |
|---|---|---|
| `MODEL_ID` | `Qwen/Qwen2.5-7B-Instruct` | The state-of-the-art instructor model used as our "teacher". |
| `QUANTIZATION` | `4-bit (NF4)` | Reduces memory usage by ~75% while maintaining high accuracy. |
| `BATCH_SIZE` | `16` | Optimized for GPU throughput. |
| `MAX_NEW_TOKENS` | `512` | Sufficient length for a 2-4 sentence financial explanation. |

In [ ]:
MODEL_ID        = "Qwen/Qwen2.5-7B-Instruct"
DATASET_ID      = "lmassaron/FinancialPhraseBank"
OUTPUT_PATH     = "FinancialPhraseBank_explained"
PUSH_TO_REPO    = False
HF_REPO_ID      = None  # Set to "your-username/dataset-name" to push to Hub
BATCH_SIZE      = 16
MAX_NEW_TOKENS  = 512

QUANTIZATION_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

LABEL_MAP = {
    0: "negative",
    1: "neutral",
    2: "positive",
}

## 2. Prompt Engineering

To get high-quality financial reasoning, we use a **System Prompt** that assigns a specific persona to the model. This anchors the model's responses in professional terminology and logical deduction rather than generic text generation.

The `build_prompt` function then wraps each headline with its known sentiment, asking the model to justify that specific classification.

In [ ]:
SYSTEM_PROMPT = (
    "You are a senior financial analyst with deep expertise in equity markets, "
    "corporate finance, and macroeconomics. "
    "Your task is to explain, in 2-4 sentences, why a financial news headline "
    "carries a specific market sentiment. "
    "Focus strictly on the financial implications: how the news affects revenue, "
    "profitability, cash flow, investor confidence, or market positioning. "
    "Be concise and precise. Do not repeat the sentence verbatim."
)

def build_prompt(sentence: str, sentiment: str) -> str:
    return (
        f"Financial news headline:\n\"{sentence}\"\n\n"
        f"This headline has been classified as **{sentiment}** sentiment "
        f"from a financial markets perspective.\n\n"
        f"Explain why, focusing on the financial implications for investors, "
        f"the company, or the broader market."
    )

## 3. Loading the Teacher Model

We load the model using `device_map="auto"`, which automatically distributes layers across available GPUs. We set `padding_side="left"` because it is required for batch inference with decoder-only models (like Qwen, Llama, or Gemma).

In [ ]:
def load_model(model_id: str):
    print(f"Loading tokenizer and model: {model_id}")
    tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side="left")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=QUANTIZATION_CONFIG,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    model.eval()
    
    device_info = getattr(model, "hf_device_map", None) or str(model.device)
    print(f"Model loaded. Device: {device_info}")
    
    return tokenizer, model

tokenizer, model = load_model(MODEL_ID)

## 4. Inference Pipeline

This section handles the core logic of converting headlines into explanations. 

1. **Template Application**: Uses the model's official chat template to ensure the instruction format matches what the model was trained on.
2. **Batch Generation**: Generates 16 explanations at once for speed.
3. **Token Slicing**: We strip the original prompt from the output to save only the newly generated explanation.

In [ ]:
def generate_explanations_batch(
    tokenizer,
    model,
    sentences: list[str],
    labels: list[int],
) -> list[str]:
    """Run inference on a batch and return explanation strings."""

    # Build chat-formatted prompts
    chats = [
        tokenizer.apply_chat_template(
            [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": build_prompt(s, LABEL_MAP[l])},
            ],
            tokenize=False,
            add_generation_prompt=True,
        )
        for s, l in zip(sentences, labels)
    ]

    inputs = tokenizer(
        chats,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,          # greedy — deterministic and fast
            temperature=1.0,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens (strip the prompt)
    input_len = inputs["input_ids"].shape[1]
    explanations = tokenizer.batch_decode(
        outputs[:, input_len:],
        skip_special_tokens=True,
    )
    return [e.strip() for e in explanations]

## 5. Dataset Processing

We iterate through the entire dataset. The generated results will be used in the next step of the workshop to fine-tune a smaller model (Gemma 3 1B) to replicate this high-level reasoning.

In [ ]:
def process_dataset(tokenizer, model, dataset) -> list[dict]:
    """Iterate over the dataset in batches, collecting results."""
    results = []
    sentences = dataset["sentence"]
    labels    = dataset["label"]
    n         = len(sentences)

    for start in tqdm(range(0, n, BATCH_SIZE), desc="Generating explanations"):
        batch_sentences = sentences[start : start + BATCH_SIZE]
        batch_labels    = labels[start    : start + BATCH_SIZE]

        explanations = generate_explanations_batch(
            tokenizer, model, batch_sentences, batch_labels
        )

        for sentence, label, explanation in zip(
            batch_sentences, batch_labels, explanations
        ):
            results.append({
                "sentence":    sentence,
                "label":       label,
                "sentiment":   LABEL_MAP[label],
                "explanation": explanation,
            })

    return results

## 6. Execution and Saving

We process every split (Train, Validation, Test) to ensure we have a complete reasoned dataset for all stages of model development.

In [ ]:
# 1. Load dataset
print(f"Loading dataset: {DATASET_ID}")
raw = load_dataset(DATASET_ID)
print(raw)

# 2. Process every split
output_splits = {}
for split_name, data in raw.items():
    print(f"\n── Processing split: '{split_name}' ({len(data)} examples) ──")
    results = process_dataset(tokenizer, model, data)
    output_splits[split_name] = Dataset.from_list(results)

output_ds = DatasetDict(output_splits)

# 3. Save locally
output_ds.save_to_disk(OUTPUT_PATH)
print(f"\nDataset saved to: {OUTPUT_PATH}")

## 7. Verification

Let's inspect a sample output to verify the quality of the financial reasoning.

In [ ]:
first_split = list(output_splits.keys())[0]
ex = output_ds[first_split][0]
print(f"\nExample from '{first_split}':")
print(f"  Sentence  : {ex['sentence']}")
print(f"  Sentiment : {ex['sentiment']} (label={ex['label']})")
print(f"  Explanation:\n    {ex['explanation']}")

## 8. Export to Hugging Face Hub

Uploading the dataset to the Hub allows for easy versioning and sharing with the community. 

In [ ]:
if HF_REPO_ID and PUSH_TO_REPO:
    print(f"Pushing to Hub: {HF_REPO_ID}")
    output_ds.push_to_hub(HF_REPO_ID)
    print("Upload complete!")